In [7]:
import os
import glob
import re
import rasterio
import numpy as np
import matplotlib.pyplot as plt
%matplotlib qt

Let's try plotting the wave heights above mean lake level as a function of time for a particular point in lake, preferrably near/on the ramp and see how it looks!

In [ ]:
TARGET_COORD = (1350.65, 246.08) # (X,Y) coordinates of the point in the lake where we want to extract data!

TARGET_DIR = "/home/anubrata/data/PROJECTS/ShallowWater/CellSize/DEM20m/20m_results/20m_ascii" # Directory where the output ascii files are located
PREFIX = "20m" # Prefix used for running the simulation

def extract_wave_data(target_dir, prefix, target_coord):

    # Get list of all the required ascii files, the htsun files!
    search_pattern = os.path.join(
        target_dir,
        f"{prefix}_htsun[0-9][0-9][0-9][0-9].asc"
    )

    file_list = glob.glob(search_pattern)

    # Handle the case when no files are found
    if len(file_list) == 0:
        print("No files are found with the specified regex syntax")
        return [], []
    
    # Make a regex object to extract the time step from the filename
    pattern = re.compile(rf"{prefix}_htsun(\d{{4}})\.asc")
    sorted_pairs =[]

    for files in file_list:
        filename = os.path.basename(files)
        match = pattern.search(filename)
        if match is not None:
            time_step = int(match.group(1))
            sorted_pairs.append((time_step, files))
    
    sorted_pairs.sort(key=lambda x:x[0])

    time_steps = []
    for pair in sorted_pairs:
        time_steps.append(pair[0] * 5) # Data is extracted at 5s interval!

    ordered_files = []
    for pair in sorted_pairs:
        ordered_files.append(pair[1])
    
    wave_amplitudes = []

    for file in ordered_files:
        with rasterio.open(file) as src:
            coords = [target_coord] # Rasterio expects a list of coordinates

            sample_gen = src.sample(coords)
            sample = next(sample_gen)
            sample_val = sample[0] # rasterio returns an array, we only need the first value!

            if sample_val == src.nodata:
                wave_amplitudes.append(0.0)
            else:
                wave_amplitudes.append(sample_val)
    
    return time_steps, wave_amplitudes

plt.figure(figsize=(12,6))

ts, hs = extract_wave_data(TARGET_DIR, PREFIX, TARGET_COORD)
plt.plot(ts, hs, linewidth=2, alpha=0.8)
plt.title(f"Wave Amplitude at {TARGET_COORD} Over Time")
plt.xlabel("Time (s)")
plt.ylabel("Wave Amplitude (m)")
plt.grid()
plt.show()

We ran simulation for different resolution of DEMs - 5m, 10m, 20m, 40m. Now let's plot all of them in a single graph and look at the difference!

In [ ]:
TARGET_COORD = (1078.3, 250.9) # (X,Y) coordinates of the point in the lake where we want to extract data!

TARGET_DIRS = {
    "40m Resolution": {
        "dir": "/home/anubrata/data/PROJECTS/ShallowWater/CellSize/DEM40m/40m_results/40m_ascii",
        "prefix": "40m" 
    },
    "20m Resolution": {
        "dir": "/home/anubrata/data/PROJECTS/ShallowWater/CellSize/DEM20m/20m_results/20m_ascii",
        "prefix": "20m"
    },
    "10m Resolution": {
        "dir": "/home/anubrata/data/PROJECTS/ShallowWater/CellSize/DEM10m/10m_results/10m_ascii",
        "prefix": "10m"
    },
    "5m Resolution": {
        "dir": "/home/anubrata/data/PROJECTS/ShallowWater/CellSize/DEM5m/5m_results/5m_ascii",
        "prefix": "5m"
    }
}

def extract_wave_data(target_dir, prefix, target_coord):

    # Get list of all the required ascii files, the htsun files!
    search_pattern = os.path.join(
        target_dir,
        f"{prefix}_htsun[0-9][0-9][0-9][0-9].asc"
    )

    file_list = glob.glob(search_pattern)

    # Handle the case when no files are found
    if len(file_list) == 0:
        print("No files are found with the specified regex syntax")
        return [], []
    
    # Make a regex object to extract the time step from the filename
    pattern = re.compile(rf"{prefix}_htsun(\d{{4}})\.asc")
    sorted_pairs =[]

    for files in file_list:
        filename = os.path.basename(files)
        match = pattern.search(filename)
        if match is not None:
            time_step = int(match.group(1))
            sorted_pairs.append((time_step, files))
    
    sorted_pairs.sort(key=lambda x:x[0])

    time_steps = []
    for pair in sorted_pairs:
        time_steps.append(pair[0] * 5) # Data is extracted at 5s interval!

    ordered_files = []
    for pair in sorted_pairs:
        ordered_files.append(pair[1])
    
    wave_amplitudes = []

    for file in ordered_files:
        with rasterio.open(file) as src:
            coords = [target_coord] # Rasterio expects a list of coordinates

            sample_gen = src.sample(coords)
            sample = next(sample_gen)
            sample_val = sample[0] # rasterio returns an array, we only need the first value!

            if sample_val == src.nodata:
                wave_amplitudes.append(0.0)
            else:
                wave_amplitudes.append(sample_val)
    
    return time_steps, wave_amplitudes

plt.figure(figsize=(12,6))

colors =["#d7191c", "#fdae61", "#abdda4", "#2b83ba"] # Different colors for the different plots to tell the grids apart

for (label, config), color in zip(TARGET_DIRS.items(), colors):
    ts, hs = extract_wave_data(config["dir"], config["prefix"], TARGET_COORD)
    plt.plot(ts, hs, label=label, linewidth=2, alpha=0.8, color=color)

plt.title(f"Wave Amplitude at {TARGET_COORD} Over Time for Different Grid Resolutions")
plt.xlabel("Time (s)")
plt.ylabel("Wave Amplitude (m)")

plt.legend()
plt.grid()
plt.show()

Quite a lot of variety in plots. Before analysing them, let's move to checking some basic volume conservation between solid and liquid phases.

In [ ]:
TARGET_DIRS = {
    "40m Resolution": {
        "dir": "/home/anubrata/data/PROJECTS/ShallowWater/CellSize/DEM40m/40m_results/40m_ascii",
        "prefix": "40m", 
        "cellsize": 40
    },
    "20m Resolution": {
        "dir": "/home/anubrata/data/PROJECTS/ShallowWater/CellSize/DEM20m/20m_results/20m_ascii",
        "prefix": "20m",
        "cellsize": 20
    },
    "10m Resolution": {
        "dir": "/home/anubrata/data/PROJECTS/ShallowWater/CellSize/DEM10m/10m_results/10m_ascii",
        "prefix": "10m",
        "cellsize": 10
    },
    "5m Resolution": {
        "dir": "/home/anubrata/data/PROJECTS/ShallowWater/CellSize/DEM5m/5m_results/5m_ascii",
        "prefix": "5m",
        "cellsize": 5
    }
}

START_STEP = 0
END_STEP = 100

def calculate_volume(filepath):
    with rasterio.open(filepath) as src:
        data = src.read(1) # Read the first band

        nodata_val = src.nodata
        if nodata_val is not None:
            valid_data = (data != nodata_val) & (data >= 0)
        else:
            valid_data = (data >= 0)

        cell_area = src.res[0] * src.res[1] 
        volume = np.sum(data[valid_data]) * cell_area

    return volume

fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True, sharey=True)

axes = axes.flatten()

for ax, (label, config) in zip(axes, TARGET_DIRS.items()):

    base_dir = config["dir"]

    timesteps = []

    p1_vol = []
    p3_vol = []

    total_vol = []

    for step in range(START_STEP, END_STEP + 1):
        step_str = f"{step:04d}"

        p1_file = os.path.join(
            base_dir,
            f"{config['prefix']}_hflow1{step_str}.asc"
        )

        p3_file = os.path.join(
            base_dir,
            f"{config['prefix']}_hflow3{step_str}.asc"
        )

        vol1 = calculate_volume(p1_file)
        vol3 = calculate_volume(p3_file)

        total = vol1 + vol3

        timesteps.append(step * 5) # Time in seconds

        p1_vol.append(vol1)
        p3_vol.append(vol3)

        total_vol.append(total)

    ax.plot(
        timesteps,
        p1_vol,
        label="Phase 1",
        linewidth=2
    )

    ax.plot(
        timesteps,
        p3_vol,
        label="Phase 3",
        linewidth=2
    )

    ax.plot(
        timesteps,
        total_vol,
        label="Total",
        linewidth=3,
        linestyle="--"
    )    

    ax.set_title(label)

    ax.grid(alpha=0.3)

    ax.set_xlabel("Time (s)")

    ax.set_ylabel("Volume ($m^3$)")  

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=4
)

plt.tight_layout()
plt.show()
  